![UTN Facultad Regional Mendoza](https://raw.githubusercontent.com/javovelez/Modelos-de-Lenguaje/main/img/logo_utn_frm.png)

# Laboratorio n° 2. Parte B: Una sola tabla de palabras

**Asignatura:** Modelos de Lenguaje
**Bloque:** 2 — Redes Neuronales Recurrentes

---

## Introducción

El modelo de lenguaje de la Clase 2 tiene **dos** matrices con una fila por palabra del vocabulario. La de entrada, `embedding.weight`, de forma `(V, 128)`, es la que convierte cada índice en un vector antes de entrar a la LSTM. La de salida, `salida.weight`, de forma `(V, 256)`, es la que puntúa cada palabra a partir del estado. Entre las dos son el **83 %** de los parámetros del modelo.

La Clase 2 cerró la sección *7.2* mostrando que esas dos tablas no valen lo mismo: la de salida aprendió la semántica del castellano —colores con colores, verbos de habla entre sí— y la de entrada no, porque la fila de salida de todas las palabras recibe gradiente en todos los pasos mientras que la de entrada solo se actualiza en los lotes donde esa palabra aparece. Y terminó diciendo que una solución habitual es **atar las dos tablas**: usar una sola matriz, que se entrena desde los dos lados, y ahorrarse la otra.

Atar cambia el cálculo de los logits: en lugar de una capa `nn.Linear` dedicada a la salida, se obtienen como `h @ embedding.weight.T + sesgo`. La literatura reporta que además de ahorrar parámetros, mejora la perplejidad. Medirlo sobre este corpus da lo contrario: a cinco épocas, atar cuesta ocho puntos de perplejidad.

Ese resultado tiene dos problemas, y los dos son de método más que de modelado. El primero es que la comparación no está controlada: el modelo atado se diferencia del de la clase en **dos** factores a la vez, y no hay manera de atribuirle la diferencia a uno solo. El segundo es que la conclusión cambia de signo según cuánto se entrene. Este laboratorio diseña el experimento que resuelve los dos problemas.

Al completar este laboratorio vas a poder:

- Implementar una capa de salida que reusa la tabla de entrada, y contar exactamente los parámetros que eso ahorra.
- Leer una curva de validación entera en vez de un número final, distinguir el valor a un presupuesto fijo del mínimo de la curva, y saber cuál de los dos se reporta.
- Reconocer cuándo una comparación mezcla dos cambios, y medir el control que los separa.
- Comparar cuatro arquitecturas por tamaño y por calidad a la vez, y no por una sola cifra.
- Explicar por qué un resultado publicado y el que medís vos pueden ser los dos ciertos, y qué tiene que reportar una comparación para que se pueda saber.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- **El laboratorio compara cuatro modelos, y dos de ellos vienen entrenados.** Son los dos que no llevan código tuyo; la cuarta celda de preparación explica cuáles y por qué. Los que entrenás son dos.
- **Para esos dos hace falta GPU, y acá no es una recomendación de comodidad.** En Colab: *Entorno de ejecución > Cambiar tipo de entorno > T4 GPU*. Son dos modelos de quince épocas —15.240 pasos cada uno— que con una T4 son del orden de **diez minutos** en total. La instancia de Colab **sin acelerador** da dos núcleos y un solo hilo de PyTorch, y sobre ella los mismos dos entrenamientos se estiman en **casi cinco horas**: no es un laboratorio lento, es un laboratorio que no termina. Antes de empezar, verificá que la celda de preparación imprima `dispositivo: cuda`.
- Cada modelo entrenado **se guarda** con `entrenar_o_cargar`, el almacén de checkpoints de las clases, en la misma carpeta de Drive. Las celdas de preparación lo envuelven en `preparar_modelo`, que es la función con la que pedís los cuatro modelos. Si la sesión se corta, lo entrenado no se pierde: al volver a ejecutar, los modelos ya terminados se cargan y el laboratorio sigue desde donde quedó.
- **No hace falta haber hecho la Parte A.** Este laboratorio se apoya enteramente en el modelo de lenguaje de la Clase 2 y trae sus propias celdas de preparación.
- Este laboratorio va después de la Clase 2, y da por vistos el modelo de lenguaje, la perplejidad, las ventanas de texto y las dos tablas de embeddings de la sección *7.2*. Las consignas te mandan a la sección correspondiente cuando hace falta.
- Las celdas de setup traen el código de la Clase 2, **con un solo cambio**: `EPOCAS` pasa de 5 a 15. El Ejercicio 2 explica por qué. Usalo sin modificar nada más: si cambiás el corpus, el split o la configuración de entrenamiento, los números dejan de ser comparables con los de la clase y con los de tus compañeros.
- **Fijá las semillas que te pide cada enunciado**, y declarálas en la configuración de `preparar_modelo`.
- Las cifras dependen de la máquina y del dispositivo. Lo que se corrige es la dirección del resultado y el argumento con el que lo explicás, no el tercer decimal.

## IMPORTANTE: qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Todas las demás celdas —enunciados, explicaciones, ejemplos provistos y encabezado— **no se tocan**.

La corrección se hace celda por celda: cada respuesta se busca en la celda donde el enunciado la pide. Si escribís en otro lado, o si movés, renombrás o borrás celdas del enunciado, esa parte de tu entrega queda sin poder corregirse.

Si querés probar algo suelto, hacelo en la misma celda de actividad o en una celda nueva que agregues, y borrala antes de entregar.

---
## Preparación

Las cuatro celdas que siguen ya vienen resueltas. No hay nada que completar en ellas, pero **hay que ejecutarlas** en orden antes de empezar, y conviene leer las presentaciones, porque definen los nombres que usan los ejercicios.

La primera importa las librerías, baja los módulos de la materia —la clase `Vocabulario` y el almacén de checkpoints, con `entrenar_o_cargar`— y deja definidos `DISPOSITIVO`, `CHECKPOINTS`, `bajar` y `n_parametros`. La segunda construye el corpus de cuentos, el vocabulario y las ventanas de la Clase 2. La tercera, el modelo, sus mediciones y el entrenador. La cuarta baja los pesos de los dos modelos del laboratorio que no vas a entrenar.

Hay una quinta celda de preparación más adelante, dentro del Ejercicio 2, que construye la tabla comparativa. Va ahí, y no acá, porque necesita los dos primeros modelos ya entrenados.

In [ ]:
# ─── Setup: imports, módulos de la materia y carpeta de checkpoints ─────────
import os, math, time, random, urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# El módulo con la clase Vocabulario es el del Laboratorio 1a, el mismo de las clases.
URL_PIPELINE = ("https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
                "/utils-v1/lab1a_pipeline.py")
if not os.path.exists("lab1a_pipeline.py"):
    urllib.request.urlretrieve(URL_PIPELINE, "lab1a_pipeline.py")

# Todo lo que el laboratorio baja sale del repositorio de la materia, que da una URL
# estable. El Drive de la cátedra queda de respaldo, con los mismos identificadores que
# usan los notebooks de las clases: es la misma copia, pero Drive corta las descargas
# públicas por cuota cuando muchos bajan el mismo archivo a la vez.
RELEASES = "https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
FUENTES = {                                   # archivo: (URL del release, ID de Drive)
    "cache_modelos.py":       (f"{RELEASES}/utils-v1/cache_modelos.py",
                               "1k5wxbSMS-_3dC8FN_F-2VN6dp8tGFCpz"),
    "cuentos_es_top5000.txt": (f"{RELEASES}/corpus-u2-v1/cuentos_es_top5000.txt",
                               "1hIeUjBuvZrNC9d0xds9vRVOky52a_0oM"),
    "lab2b_clase.pt":         (f"{RELEASES}/utils-v1/lab2b_clase.pt", None),
    "lab2b_control.pt":       (f"{RELEASES}/utils-v1/lab2b_control.pt", None),
}


def bajar(nombre, destino=None):
    """Baja un archivo del repositorio de la materia; si no responde, del Drive de la cátedra."""
    destino = destino or nombre
    if os.path.exists(destino):
        return
    url, id_drive = FUENTES[nombre]
    try:
        urllib.request.urlretrieve(url, destino)
    except Exception as falla:
        if id_drive is None:                  # sin copia en el Drive de la cátedra
            print(f"no se pudo bajar {nombre} ({falla}).")
            return
        print(f"el repositorio de la materia no respondió ({falla}); se usa el Drive.")
        try:
            import gdown
        except ImportError:
            os.system("pip install -q gdown")
            import gdown
        gdown.download(id=id_drive, output=destino, quiet=False)


bajar("cache_modelos.py")

from lab1a_pipeline import Vocabulario
from cache_modelos import preparar_carpeta, entrenar_o_cargar, firma_de

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINTS = preparar_carpeta()  # la misma carpeta de checkpoints que las clases


def n_parametros(m):
    """Cantidad total de parámetros de un modelo."""
    return sum(p.numel() for p in m.parameters())


pd.Series({"torch": torch.__version__, "dispositivo": DISPOSITIVO})

### El corpus de cuentos, el vocabulario y las ventanas

La celda que sigue es la Parte 1 y la Parte 2 de la Clase 2, sin cambios: el corpus de cuentos recortado a las 5.000 palabras más frecuentes, el split **por oración** —se permutan las oraciones con una semilla fija y el primer 5 % va a validación—, y después cada partición pegada en una sola secuencia larga de índices, que es lo que las ventanas recorren.

| Nombre | Qué es |
|---|---|
| `vocab`, `V` | el vocabulario de los cuentos y su tamaño, 5.003 |
| `ids_ent`, `ids_val` | el corpus entero como una secuencia de índices, entrenamiento y validación |
| `VentanasDeTexto` | el `Dataset` de ventanas de `L + 1` tokens: `x` son los primeros `L`, `y` los últimos `L` |
| `dl_ent` | el `DataLoader` de entrenamiento |
| `L`, `LOTE` | 32 y 64 |

In [ ]:
# ─── Setup: el corpus de cuentos, el vocabulario y las ventanas ─────────────
bajar("cuentos_es_top5000.txt")

texto = open("cuentos_es_top5000.txt", encoding="utf-8").read()
oraciones = [o.rstrip(" .") + " ." for o in texto.split(" . ")]
vocab = Vocabulario(oraciones, tokenizador=str.split)
V = len(vocab)

# Split por oración, y después todo pegado en una sola secuencia (Clase 2, sección 2.2).
perm = torch.randperm(len(oraciones), generator=torch.Generator().manual_seed(0))
N_VAL = len(oraciones) // 20
oraciones_val = [oraciones[i] for i in perm[:N_VAL]]
oraciones_ent = [oraciones[i] for i in perm[N_VAL:]]
ids_ent = torch.tensor([vocab[t] for o in oraciones_ent for t in o.split()])
ids_val = torch.tensor([vocab[t] for o in oraciones_val for t in o.split()])

L, LOTE = 32, 64


class VentanasDeTexto(Dataset):
    """Ventanas consecutivas de L+1 tokens: x son los primeros L, y los últimos L."""

    def __init__(self, ids, L):
        self.ids, self.L = ids, L

    def __len__(self):
        return (len(self.ids) - 1) // self.L

    def __getitem__(self, i):
        ventana = self.ids[i * self.L: i * self.L + self.L + 1]
        return ventana[:-1], ventana[1:]


dl_ent = DataLoader(VentanasDeTexto(ids_ent, L), batch_size=LOTE,
                    shuffle=True, drop_last=True)

pd.Series({"tokens de entrenamiento": len(ids_ent), "tokens de validación": len(ids_val),
           "vocabulario": V, "pasos por época": len(dl_ent)})

### El modelo de lenguaje de la Clase 2, sus mediciones y el entrenador

| Nombre | Qué es |
|---|---|
| `ModeloLenguaje` | `Embedding → LSTM → Linear(V)`, con logits en **cada** posición de la secuencia |
| `demo` | una instancia sin entrenar, para inspeccionar la arquitectura |
| `perdida_lm` | la entropía cruzada sobre las `B · L` predicciones del lote |
| `perplejidad` | su exponencial, ventana por ventana, sobre una secuencia entera de índices |
| `muestrear` | genera tokens muestreando de la *softmax*, realimentando el token y el estado |
| `entrenador` | el bucle de entrenamiento de la Clase 2: entrena `EPOCAS` pasadas y devuelve `{"historia", "muestras"}` |
| `ids_monitor` | el tramo de validación sobre el que el entrenador construye la curva |
| `vecinos`, `PALABRAS` | los vecinos por coseno de una tabla `(V, d)` y las diez palabras de sondeo de la sección *7.1* |
| `DIM`, `OCULTA`, `DROPOUT` | 128, 256 y 0.0 |
| `EPOCAS`, `LR`, `CADA` | 15, 2e-3 y 250 |

**El único valor que cambia respecto de la Clase 2 es `EPOCAS`**, que pasa de 5 a 15. Con 1.016 pasos por época, eso son 15.240 pasos, y como `entrenador` mide cada `CADA = 250` pasos, cada entrenamiento deja una curva de 61 puntos. El Ejercicio 2 explica para qué.

`entrenador` lee `EPOCAS` y `dl_ent` del entorno, así que basta con llamarlo como `entrenador` —sin paréntesis— al pasárselo a `preparar_modelo`.

In [ ]:
# ─── Setup: el modelo de lenguaje de la Clase 2 y su entrenamiento ──────────
DIM, OCULTA, DROPOUT = 128, 256, 0.0
EPOCAS, LR, CADA = 15, 2e-3, 250          # la Clase 2 entrenó 5 épocas; acá son 15
ids_monitor = ids_val[:40_000]            # un tramo fijo de validación, para la curva


class ModeloLenguaje(nn.Module):
    """Embedding → LSTM → Linear. Devuelve V logits EN CADA POSICIÓN de la secuencia."""

    def __init__(self, n_vocab=None, dim_emb=DIM, dim_oculta=OCULTA, dropout=DROPOUT):
        super().__init__()
        n_vocab = n_vocab or V
        self.embedding = nn.Embedding(n_vocab, dim_emb)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(dim_emb, dim_oculta, batch_first=True)
        self.salida = nn.Linear(dim_oculta, n_vocab)

    def forward(self, x, estado=None):
        e = self.dropout(self.embedding(x))
        h, estado = self.lstm(e, estado)
        return self.salida(self.dropout(h)), estado


def perdida_lm(logits, y):
    """Entropía cruzada sobre las B·L predicciones del lote."""
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))


@torch.no_grad()
def perplejidad(m, ids, L=L, lote=256):
    """Exponencial de la entropía cruzada media, ventana por ventana (Clase 2, sección 3.1)."""
    m.eval()
    nll, n = 0.0, 0
    for x, y in DataLoader(VentanasDeTexto(ids, L), batch_size=lote):
        logits, _ = m(x.to(DISPOSITIVO))
        nll += perdida_lm(logits, y.to(DISPOSITIVO)).item() * y.numel()
        n += y.numel()
    m.train()
    return math.exp(nll / n)


@torch.no_grad()
def muestrear(m, semilla, n=20, seed=0):
    """Genera n tokens muestreando de la softmax, realimentando el token y el estado."""
    m.eval()
    gen = torch.Generator().manual_seed(seed)
    ids = [vocab[t] for t in semilla.split()]
    logits, estado = m(torch.tensor([ids], device=DISPOSITIVO))
    for _ in range(n):
        p = torch.softmax(logits[0, -1], -1).cpu()
        siguiente = torch.multinomial(p, 1, generator=gen).item()
        ids.append(siguiente)
        logits, estado = m(torch.tensor([[siguiente]], device=DISPOSITIVO), estado)
    m.train()
    return " ".join(vocab.decodificar(ids))


def entrenador(m):
    """Entrena EPOCAS pasadas sobre las ventanas. Devuelve la historia y las muestras."""
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    historia, muestras = [], {}

    def medir(paso, ppl_ent):
        ppl_val = perplejidad(m, ids_monitor)
        muestras[paso] = muestrear(m, "la princesa")
        historia.append({"paso": paso, "ppl_ent": ppl_ent, "ppl_val": ppl_val})
        print(f"   paso {paso:5d}  ppl ent {ppl_ent:7.1f}  ppl val {ppl_val:6.1f} | "
              f"{muestras[paso][:70]}", flush=True)

    medir(0, float("nan"))
    paso, acumulada = 0, 0.0
    for epoca in range(EPOCAS):
        m.train()
        for x, y in dl_ent:
            x, y = x.to(DISPOSITIVO), y.to(DISPOSITIVO)
            logits, _ = m(x)
            perdida = perdida_lm(logits, y)
            opt.zero_grad()
            perdida.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            paso += 1
            acumulada += perdida.item()
            if paso % CADA == 0:
                medir(paso, math.exp(acumulada / CADA))
                acumulada = 0.0
    return {"historia": historia, "muestras": muestras}


def vecinos(tabla, palabra, n=6):
    """Las n palabras cuya fila de la tabla apunta en la dirección más parecida."""
    E = F.normalize(tabla.detach().cpu(), dim=1)
    similitud = E @ E[vocab[palabra]]
    return ", ".join(vocab.itos[i] for i in similitud.topk(n + 1).indices[1:])


PALABRAS = ["rey", "princesa", "él", "tres", "dijo", "bosque", "hermoso", "nunca", "madre", "rojo"]

demo = ModeloLenguaje().to(DISPOSITIVO)
demo

### Los pesos de los dos modelos que no escribís vos

El laboratorio compara cuatro modelos, y **dos de ellos no llevan una línea de código tuya**: el modelo de lenguaje de la Clase 2, que es el punto de comparación de todo lo demás, y el control del Ejercicio 3, que es ese mismo `ModeloLenguaje` de la celda anterior construido con otros dos argumentos. Esos dos **vienen entrenados**. La celda que sigue descarga sus pesos y los deja en la carpeta de *checkpoints*: son 22 MB, y se descargan una sola vez.

Los otros dos —`atado256` en el Ejercicio 2 y `proyectado` en el Ejercicio 4— los entrenás vos, porque los dos ejercitan la clase que escribís en el Ejercicio 1.

Los cuatro modelos del laboratorio se piden con **`preparar_modelo`**, que define la celda que sigue: le pasás un nombre, una función que instancia el modelo, el entrenador y la configuración, y decide. Si hay un checkpoint con ese nombre y su **firma** coincide con la configuración, carga los pesos y no entrena; si no hay checkpoint, entrena.

La firma combina el vocabulario con la configuración, así que un solo valor mal declarado —un `dim` que no corresponde, un `arq` mal escrito— alcanza para que deje de coincidir, y ahí está el riesgo: pedir un modelo provisto con la configuración equivocada no es un error de Python, es un entrenamiento de quince épocas que nadie pidió y que al terminar sobreescribe los pesos descargados. Por eso `preparar_modelo` no entrena en ese caso: imprime qué valor difiere y corta la celda.

```
[lab2b_clase] la configuración no coincide con la del checkpoint: no se entrena nada
    dim: el checkpoint dice 128 y le pasaste 256
ValueError: lab2b_clase: revisá la configuración contra las celdas de preparación.
```

Si aparece, corregí ese valor contra las celdas de preparación: los valores de la configuración salen de ahí y no de números escritos a mano. Y si lo que querés es entrenar uno de los dos por tu cuenta, borrá su `.pt` de la carpeta de *checkpoints* antes de ejecutar la celda. Sin acelerador cada uno son unas dos horas y media, y ninguna conclusión del laboratorio cambia.

In [ ]:
# ─── Setup: los pesos de los dos modelos que no escribís vos ────────────────
# El de la Clase 2 y el control son el ModeloLenguaje de la celda anterior con otros
# argumentos: no hay nada tuyo adentro, así que vienen entrenados. Los dos que salen
# de tu clase atada los entrenás vos, en los Ejercicios 2 y 4.
PESOS_PROVISTOS = ["lab2b_clase.pt", "lab2b_control.pt"]
for archivo in PESOS_PROVISTOS:
    bajar(archivo, os.path.join(CHECKPOINTS, archivo))


def preparar_modelo(nombre, construir, entrenar, vocab, **config):
    """`entrenar_o_cargar` con un chequeo previo. Los cuatro modelos pasan por acá.

    Carga si hay checkpoint y su firma coincide con la configuración; entrena si no
    hay checkpoint. Si hay checkpoint y la firma NO coincide, en lugar de entrenar
    quince épocas que nadie pidió, dice qué valor difiere y corta la celda.
    """
    ruta = os.path.join(CHECKPOINTS, f"{nombre}.pt")
    if os.path.exists(ruta):
        guardado = torch.load(ruta, map_location="cpu", weights_only=False)
        if guardado.get("firma") != firma_de(vocab, **config):
            guardada = guardado.get("config", {})
            print(f"[{nombre}] la configuración no coincide con la del checkpoint: "
                  f"no se entrena nada")
            for clave in sorted(set(guardada) | set(config)):
                if guardada.get(clave) != config.get(clave):
                    print(f"    {clave}: el checkpoint dice "
                          f"{guardada.get(clave, '(no está)')!r} y le pasaste "
                          f"{config.get(clave, '(no está)')!r}")
            if guardada == config:
                print("    los valores coinciden, así que lo que cambió es el vocabulario")
            raise ValueError(
                f"{nombre}: revisá la configuración contra las celdas de preparación. "
                f"Si lo que querés es entrenarlo vos, borrá {ruta} y volvé a ejecutar.")
    return entrenar_o_cargar(nombre, construir, entrenar, vocab, **config)


pd.Series({archivo: f"{os.path.getsize(os.path.join(CHECKPOINTS, archivo)) / 1e6:.1f} MB"
           for archivo in PESOS_PROVISTOS})

### Ejercicio 1 — Una sola tabla

**Objetivo:** Escribir un modelo de lenguaje que use la tabla de entrada también como tabla de salida, y contar exactamente qué parámetros aparecen y cuáles desaparecen.

**Enunciado:**

> **Nota — qué quiere decir atar las tablas.** La capa `nn.Linear(dim_oculta, V)` de `ModeloLenguaje` guarda una matriz `salida.weight` de forma `(V, dim_oculta)`: la fila `i` es el vector con el que se puntúa la palabra `i`, y el logit de esa palabra es el producto interno entre esa fila y el estado `h`. La tabla de embeddings `embedding.weight` también tiene una fila por palabra, de forma `(V, dim_emb)`. **Atarlas es usar una sola matriz para las dos cosas**: calcular los logits como `h @ embedding.weight.T + sesgo`, sin una capa `nn.Linear` dedicada a la salida. El sesgo sí hace falta y queda como un `nn.Parameter` de `V` números, porque atar las matrices no ata los sesgos: no hay ningún sesgo del lado de la entrada.
>
> El producto `h @ embedding.weight.T` solo está definido si `h` tiene tantas columnas como la tabla, o sea si `dim_oculta == dim_emb`. Cuando las dos dimensiones no coinciden hay que adaptar una de las dos, y la opción que menos parámetros agrega es proyectar el estado con una `nn.Linear(dim_oculta, dim_emb)` antes de multiplicar. La clase de este ejercicio tiene que contemplar los dos casos; el Ejercicio 4 usa el segundo.

1. **Escribí la clase `ModeloLenguajeAtado`**, que herede de `ModeloLenguaje` y reciba los mismos argumentos. En su `__init__`, después de llamar al de la clase madre:

   - **Borrá la capa de salida heredada**, con `del self.salida`.
   - **Creá `self.proyeccion`**, que es la que adapta el estado al ancho de la tabla de embeddings. Cuál de las dos formas de la Nota le toca depende de los argumentos con los que se construya el modelo, así que **el `__init__` la elige con un condicional**: si `dim_oculta` y `dim_emb` son distintas, una `nn.Linear(dim_oculta, dim_emb)`; si son iguales, una `nn.Identity()`, que no tiene parámetros y devuelve el tensor sin tocarlo. Resuelto ahí, el `forward` aplica `self.proyeccion` siempre, sin preguntar nada.
   - **Creá `self.sesgo`**, un `nn.Parameter` de `V` ceros.

   Después **sobrescribí `forward`** para que aplique el embedding, la LSTM y la proyección, y devuelva `h @ self.embedding.weight.T + self.sesgo` junto con el estado, con el mismo `self.dropout` en los mismos dos lugares que `ModeloLenguaje`.

2. **Agregale esta propiedad**, tal cual, dentro de la clase:

   ```python
   @property
   def salida(self):
       return self.embedding
   ```

   No agrega ningún parámetro: `salida` deja de ser una capa y pasa a ser un segundo nombre para `self.embedding`. Con eso, la expresión `m.salida.weight` devuelve la matriz con la que se calculan los logits en los cuatro modelos del laboratorio: en los de dos tablas es la de la capa de salida, y en los atados es la de embeddings, que cumple las dos funciones. El Ejercicio 4 usa esa expresión para recorrer las cuatro variantes con una sola línea de código.

3. **Creá un modelo atado** con `dim_emb=256` y `dim_oculta=256`, y guardalo en una variable llamada `atado256`. Llamá a `torch.manual_seed(0)` en la línea inmediatamente anterior a la construcción, y llevá el modelo a `DISPOSITIVO` con `.to()`. Esta es la forma directa de atar: agrandar el embedding hasta el tamaño del estado, para que el producto se pueda calcular sin ninguna capa en el medio.

4. **Comprobá que no quedó una segunda tabla.** Imprimí, para `demo` y para `atado256`, la lista de claves de `state_dict()` y la cantidad de parámetros. En el atado no tiene que aparecer ninguna clave `salida.weight`. Pasale además a `atado256` un lote de `dl_ent` e imprimí la forma de los logits, que tiene que ser `(64, 32, 5003)`.

> **Pista:** `del self.salida` funciona sobre un `nn.Module` y saca la capa del registro de submódulos, así que deja de aparecer en `state_dict()` y en `parameters()`. La propiedad del punto 2 puede convivir con el `self.salida = nn.Linear(...)` que ejecuta el `__init__` heredado, porque `nn.Module` intercepta las asignaciones de submódulos antes de que Python consulte la propiedad.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) `atado256` tiene **menos** parámetros que el modelo de la clase, aunque su tabla de embeddings es del doble de ancho. Explicá de dónde sale la diferencia con la cuenta término por término: qué crece al pasar `dim_emb` de 128 a 256, qué desaparece y qué aparece. Las cuatro cifras tienen que sumar exactamente la diferencia que imprimiste.

b) La Clase 2, sección *7.2*, midió que la fila de salida de **todas** las palabras recibe gradiente en cada uno de los 1.016 pasos de una época, mientras que la fila de entrada de una palabra solo se actualiza en los lotes donde esa palabra aparece. Explicá qué le pasa a ese conteo cuando las dos tablas son una sola, y por qué eso es un argumento a favor de atar que no tiene nada que ver con ahorrar memoria.

*(Escribí tu respuesta acá)*

### Ejercicio 2 — Quince épocas, y dos presupuestos en una sola curva

**Objetivo:** Comparar el modelo de la clase y el atado bajo el mismo presupuesto de entrenamiento, y leer el resultado a dos presupuestos distintos sin entrenar cuatro veces.

**Enunciado:**

El **presupuesto de entrenamiento** de una variante es la cantidad de épocas que se le concede, y es parte del montaje experimental: dos variantes solo son comparables si corrieron con el mismo.

La propuesta de la que sale este laboratorio midió las variantes a cinco épocas y dejó abierta la pregunta de qué pasa con más entrenamiento. Responderla parece costar el doble de entrenamientos, y no: **`entrenador` registra la perplejidad de validación cada 250 pasos**, así que una sola corrida de quince épocas contiene el resultado a cinco y el resultado a quince. Una época son 1.016 pasos, así que el paso 5.000 es aproximadamente la época 5 y el 15.000 es la 15.

1. **Cargá el modelo de la clase entrenado quince épocas.** Es el que bajó la celda de preparación, y la llamada es la misma que si lo entrenaras: `preparar_modelo` con el nombre `"lab2b_clase"`, un constructor que fije la semilla de PyTorch en 0 y devuelva un `ModeloLenguaje()` en `DISPOSITIVO`, `entrenador` como entrenador y `vocab` como vocabulario. **Constructor** acá no es el `__init__` de la clase que escribiste en el Ejercicio 1, sino el segundo argumento de `preparar_modelo`: una función aparte, sin argumentos, que instancia el modelo y lo devuelve sin entrenar. Es donde va fijada la semilla, porque `preparar_modelo` la llama tanto si entrena como si carga. En la configuración declará `arq="ModeloLenguaje"`, `L`, `dim=DIM`, `oculta=OCULTA`, `dropout=DROPOUT`, `epocas=EPOCAS`, `lote=LOTE`, `lr=LR`, `n_ent=len(ids_ent)` y `seed=0`. **Esa configuración es la que tiene que coincidir con la del checkpoint bajado**, y por eso los valores salen de las celdas de preparación y no de números escritos a mano. El modelo y sus extras tienen que quedar en `lm_clase` y `extras_clase`.

2. **Entrená `atado256` quince épocas**, igual, con el nombre `"lab2b_atado256"`, `arq="ModeloLenguajeAtado"`, `dim=256` y `oculta=256`. Quedan en `lm_atado256` y `extras_atado256`. Este es el primero de los dos modelos que entrenás vos.

3. Imprimí la perplejidad de los dos sobre `ids_val` **entero**.

> **Nota:** la perplejidad de la curva y la de `perplejidad(m, ids_val)` **no son la misma medición**. El entrenador mide sobre `ids_monitor`, los primeros 40.000 tokens de validación, para que medir cada 250 pasos no cueste más que entrenar; `perplejidad(m, ids_val)` mide sobre los 109.003 tokens completos, que es lo que reportó la Clase 2. Las dos difieren en menos de un punto, pero la comparable con el **73,1** de la clase es la segunda.

> **Nota:** el modelo de la clase se carga, así que de esa primera llamada vas a ver un solo mensaje, con la fecha en que se entrenó y cuánto tardó. `atado256` sí se entrena e imprime 61 líneas, una por medición: es esperable y no requiere ninguna intervención. Lo que la primera llamada **no** puede hacer es entrenar: si la configuración que declaraste no coincide con la del checkpoint, `preparar_modelo` corta la celda y te dice qué valor difiere.

In [ ]:
# Tu código aquí

### La tabla comparativa del laboratorio

`extras["historia"]` es la curva que devolvió cada entrenamiento, una lista de diccionarios con `paso`, `ppl_ent` y `ppl_val`, uno cada 250 pasos, más el punto del paso 0, que corresponde al modelo sin entrenar y se descarta.

La celda que sigue ya viene resuelta, como las de la Preparación, y **hay que ejecutarla**. Lee esas curvas y construye **`comparacion`**, la tabla sobre la que se analiza el resto del laboratorio, con una columna por modelo y ocho filas.

| Fila | Qué es |
|---|---|
| `parámetros` | la cantidad de parámetros del modelo |
| `5 ép.`, `15 ép.` | la perplejidad de validación en el punto registrado más cercano a la época 5 y a la época 15 |
| `mínimo` | el valor más bajo de toda la curva de validación |
| `época del mínimo` | en qué época se alcanzó ese mínimo |
| `ppl val. entera` | la perplejidad del modelo entrenado sobre `ids_val` entero, no sobre `ids_monitor` |
| `ppl ent. final` | la perplejidad de **entrenamiento** del último punto de la curva |
| `brecha` | la diferencia entre las dos anteriores |

Las cuatro primeras filas las calcula `resumen_de_curva(historia)`, y la columna entera, `fila_de(modelo, extras)`. Las dos funciones quedan definidas para los Ejercicios 3 y 4, que le agregan a `comparacion` una columna cada uno. La celda deja además `PASOS_POR_EPOCA`, los 1.016 pasos de una época, que es lo que convierte un número de paso en un número de época.

> **Nota:** la `brecha` mide cuánto **sobreajustó** el modelo, o sea cuánto mejor le va sobre el texto que vio durante el entrenamiento que sobre el que no. Los Ejercicios 3 y 4 la usan.

In [ ]:
# ─── Setup: leer una curva y construir la tabla comparativa ─────────────────
PASOS_POR_EPOCA = len(dl_ent)


def resumen_de_curva(historia, pasos_por_epoca=None):
    """De una curva de validación: el valor a 5 y a 15 épocas, el mínimo y su época."""
    pasos_por_epoca = pasos_por_epoca or PASOS_POR_EPOCA
    puntos = [f for f in historia if f["paso"] > 0]
    en_epoca = lambda e: min(puntos, key=lambda f: abs(f["paso"] - e * pasos_por_epoca))
    mejor = min(puntos, key=lambda f: f["ppl_val"])
    return {"5 ép.": round(en_epoca(5)["ppl_val"], 1),
            "15 ép.": round(en_epoca(15)["ppl_val"], 1),
            "mínimo": round(mejor["ppl_val"], 1),
            "época del mínimo": round(mejor["paso"] / pasos_por_epoca, 1)}


def fila_de(m, extras):
    """Una columna de la tabla comparativa: tamaño, curva y cuánto sobreajustó."""
    ppl_val = round(perplejidad(m, ids_val), 1)
    ppl_ent = round(extras["historia"][-1]["ppl_ent"], 1)
    return {"parámetros": n_parametros(m),
            **resumen_de_curva(extras["historia"]),
            "ppl val. entera": ppl_val,
            "ppl ent. final": ppl_ent,
            "brecha": round(ppl_val - ppl_ent, 1)}


comparacion = pd.DataFrame({"clase": fila_de(lm_clase, extras_clase),
                            "atado256": fila_de(lm_atado256, extras_atado256)})
comparacion

**Pregunta de análisis:**

a) Escribí la conclusión sobre el atado de pesos que sacaría alguien que hubiera entrenado **solo cinco épocas**, y la que sale de la tabla de la celda anterior leyendo la fila de quince. Decí con cuántos puntos de perplejidad se sostiene cada una.

b) La tabla compara `atado256` contra el modelo de la clase. ¿Esa comparación aísla el efecto de atar las tablas? Enumerá **todas** las diferencias entre los dos modelos y decí, para cada una, si podés descartarla como causa de la diferencia de perplejidad.

*(Escribí tu respuesta acá)*

### Ejercicio 3 — El control que separa los dos cambios

**Objetivo:** Sumar el modelo que falta para poder atribuirle la diferencia al atado y no al tamaño, y leer los dos efectos por separado.

**Enunciado:**

Un **control** es una corrida que difiere del modelo de interés en exactamente un factor, el que se quiere medir. Acá hace falta un modelo con las dos tablas sueltas pero con el mismo tamaño que `atado256`, o sea con `dim_emb` y `dim_oculta` en 256. Con él, la diferencia contra `atado256` es solo el atado, y la diferencia contra el modelo de la clase es solo el tamaño.

Es el modelo más grande de los tres: 3.092.875 parámetros, porque tiene la tabla de embeddings ancha **y** la capa de salida.

1. **Cargalo** con `preparar_modelo`, bajo el nombre `"lab2b_control"`. Es el segundo de los dos que la celda de preparación bajó entrenados, por el mismo motivo que el de la clase: es el `ModeloLenguaje` del setup con otros dos argumentos, y no contiene código escrito por vos. El constructor tiene que fijar la semilla en 0 y devolver un `ModeloLenguaje(dim_emb=256, dim_oculta=256)` en `DISPOSITIVO`. En la configuración, `arq="ModeloLenguaje"`, `dim=256`, `oculta=256`, y el resto igual que en el Ejercicio 2. Quedan en `lm_control` y `extras_control`.

2. **Agregale la columna a la tabla.** Volvé a construir `comparacion` con las tres columnas —`clase`, `atado256` y `control (suelto256)`— con `fila_de`, la función que define la celda de preparación del Ejercicio 2.

> **Nota — cuál de los tres presupuestos se reporta.** El Ejercicio 2 dejó tres lecturas del mismo par de corridas, y no son tres opciones entre las que se elige la que convenga. Cinco y quince épocas son cortes arbitrarios: nadie entrena un modelo hasta una época fijada de antemano. Lo que se hace es **cortar donde la validación deja de bajar**, porque entrenar más allá de ese punto empeora el modelo que vas a usar. El mínimo de la curva es entonces la perplejidad que la variante realmente te da, y es la cifra con la que se compara.
>
> Las columnas de cinco y quince épocas no sobran: son las que muestran **cuánto se distorsiona la conclusión** cuando se reporta un corte fijo en vez del mínimo. Pero son el error, no la alternativa.

> **Nota:** el nombre del checkpoint importa. Los tres notebooks de teoría y los dos laboratorios de esta unidad comparten la misma carpeta de Drive, y la firma que combina el vocabulario con la configuración es lo único que decide si un checkpoint sirve. Dos modelos distintos guardados bajo el mismo nombre y la misma configuración son un modelo cargado dos veces, sin ningún aviso.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Con las tres columnas, separá los dos efectos —cuántos puntos de perplejidad aporta agrandar el embedding de 128 a 256 sin atar, y cuántos aporta atar a igual tamaño— en los **tres** presupuestos de la tabla: a cinco épocas, a quince y en el mínimo de cada curva. Seis cifras, con su signo. Decí cuál de las tres lecturas es la que hay que reportar y por qué.

b) El control es el modelo más grande de los tres y a quince épocas es el peor. Explicá qué le pasa usando las filas `ppl ent. final` y `brecha`, y por qué su `época del mínimo` es la más temprana de las tres.

*(Escribí tu respuesta acá)*

### Ejercicio 4 — Atar sin agrandar

**Objetivo:** Sacar del experimento el cambio que no se quería —agrandar el embedding— y comparar las cuatro variantes por tamaño y por calidad a la vez.

**Enunciado:**

El Ejercicio 3 mostró que agrandar la tabla de 128 a 256 columnas empeora la perplejidad a quince épocas. La cuarta variante evita ese cambio: deja el embedding en 128, y adapta el estado de 256 al tamaño de la tabla con la `nn.Linear(256, 128)` que tu `ModeloLenguajeAtado` ya crea sola cuando las dimensiones difieren. Es la más pequeña de las cuatro: la tabla de salida desaparece y la de entrada no crece.

**Parte A — El cuarto entrenamiento.**

1. **Entrená `proyectado` quince épocas** con `preparar_modelo`, bajo el nombre `"lab2b_proyectado"`. Es el segundo y último entrenamiento del laboratorio. El constructor fija la semilla en 0 y devuelve un `ModeloLenguajeAtado(dim_emb=DIM, dim_oculta=OCULTA)` en `DISPOSITIVO`. En la configuración, `arq="ModeloLenguajeAtado"`, `dim=DIM`, `oculta=OCULTA`, y el resto igual. Quedan en `lm_proyectado` y `extras_proyectado`.

2. Imprimí su cantidad de parámetros y qué fracción es de la del modelo de la clase.

In [ ]:
# Tu código aquí

### Las cuatro variantes juntas

La celda que sigue ya viene resuelta, como `setup-curva`, y **hay que ejecutarla**. Reúne las cuatro variantes y cierra la comparación del laboratorio.

| Nombre | Qué es |
|---|---|
| `modelos` | las cuatro variantes en orden —`"clase"`, `"atado256"`, `"control (suelto256)"` y `"proyectado"`—, cada una como la tupla `(modelo, extras)` |
| `comparacion` | la tabla de `setup-curva`, ahora con las cuatro columnas |

Grafica además las cuatro curvas de validación en un solo eje, con la **época** en el eje horizontal —el paso dividido por `PASOS_POR_EPOCA`— y la perplejidad de validación en el vertical. El eje vertical va de 68 a 120 porque el modelo sin entrenar y las primeras mediciones están dos órdenes de magnitud más arriba y comprimirían la zona donde se deciden las diferencias.

El gráfico y la tabla son con lo que se responden las dos preguntas de este ejercicio y el análisis final del Ejercicio 5.

In [ ]:
# ─── Setup: las cuatro curvas en un eje y la tabla final ────────────────────
modelos = {"clase": (lm_clase, extras_clase),
           "atado256": (lm_atado256, extras_atado256),
           "control (suelto256)": (lm_control, extras_control),
           "proyectado": (lm_proyectado, extras_proyectado)}

plt.figure(figsize=(7.5, 4.5))
for etiqueta, (m, extras) in modelos.items():
    puntos = [f for f in extras["historia"] if f["paso"] > 0]
    plt.plot([f["paso"] / PASOS_POR_EPOCA for f in puntos],
             [f["ppl_val"] for f in puntos], label=etiqueta)
plt.ylim(68, 120)
plt.xlabel("época")
plt.ylabel("perplejidad de validación")
plt.title("Las cuatro variantes, quince épocas")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

comparacion = pd.DataFrame({etiqueta: fila_de(m, extras)
                            for etiqueta, (m, extras) in modelos.items()})
comparacion

**Parte B — Leer la tabla de palabras, no solo la perplejidad.**

La sección *7.1* de la Clase 2 leyó las tablas de embeddings por **vecinos por coseno**: las palabras cuya fila apunta en la dirección más parecida. Esa lectura es la que mostró que la tabla de salida había aprendido la semántica del castellano. `vecinos(tabla, palabra)` está en las celdas de preparación, y la propiedad `salida` que escribiste en el Ejercicio 1 hace que `m.salida.weight` sea la tabla que produce los logits en los cuatro modelos.

3. **Construí un `DataFrame` con los vecinos** de `"rey"`, `"dijo"` y `"rojo"` en la tabla de salida de las cuatro variantes, recorriendo el diccionario `modelos` de la celda anterior: una columna por modelo, una fila por palabra.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) `proyectado` es el más pequeño de los cuatro y el que alcanza la perplejidad más baja a quince épocas. Explicá por qué, usando tres cifras de la tabla: su cantidad de parámetros, su `ppl ent. final` y su `brecha`. Decí además por qué eso **no** quiere decir que sea el mejor modelo de lenguaje de los cuatro, observando la fila `mínimo`.

b) `atado256` y `proyectado` tienen los dos una sola tabla, y sin embargo los vecinos de `atado256` son los más coherentes semánticamente de los cuatro y los de `proyectado` los menos. La única diferencia entre los dos es la `Linear(256, 128)`. Explicá por qué esa capa hace que los vecinos por coseno de `embedding.weight` dejen de decir lo que dicen en las otras tres variantes.

> **Nota para b):** con la proyección, el logit de la palabra $i$ es $(h W^\top) \cdot E_i$, que es lo mismo que $h \cdot (W^\top E_i)$. Escribilo así y preguntate cuál es, entonces, el vector que efectivamente puntúa a la palabra $i$.

*(Escribí tu respuesta acá)*

### Ejercicio 5 — Qué hay que reportar para que se pueda saber

**Objetivo:** Cerrar el laboratorio con lo que quedó demostrado sobre el método, que es más de lo que quedó demostrado sobre el atado de pesos.

**Enunciado:**

> **Nota — qué dice la literatura.** El atado de pesos se propuso en dos trabajos presentados en 2017: *Using the Output Embedding to Improve Language Models*, de Press y Wolf (EACL), y *Tying Word Vectors and Word Classifiers: A Loss Framework for Language Modeling*, de Inan, Khosravi y Socher (ICLR). Los dos reportan que atar **mejora** la perplejidad además de ahorrar parámetros, y el argumento es el del Ejercicio 1: una sola representación por palabra, entrenada desde los dos lados, con menos parámetros que sobreajustar.
>
> Press y Wolf miden sobre *Penn Treebank*, y su tabla **sin** *dropout* es el régimen de este laboratorio:
>
> | PTB, sin *dropout* | parámetros | ppl val. | ppl test |
> |---|---|---|---|
> | modelo base | 4,65 M | 120,7 | 114,5 |
> | base + atado | 2,65 M | 117,5 | 112,4 |
> | base + proyección penalizada | 4,69 M | 116,0 | 111,7 |
> | base + atado + proyección penalizada | 2,69 M | **104,9** | 100,9 |
>
> Inan et al., con LSTM variacional y *dropout*, mejoran de 92,6 a 89,9 en validación sobre PTB con el modelo pequeño y de 76,8 a 72,5 con el grande. O sea que atar les paga **con y sin regularización**.
>
> Las dos últimas filas son `proyectado`. Press y Wolf insertan una matriz `P` antes de la tabla de salida, calculan los logits como `V P h`, y le suman a la pérdida el término `λ‖P‖₂` con `λ = 0,15`. Lo llaman **regularización de la proyección**, y dicen que sirve «para usar el mismo embedding con alguna adaptación que está bajo regularización, por lo que es especialmente adecuada para el atado». Es la variante del Ejercicio 4 de este laboratorio, **menos el término `λ‖P‖₂`**: acá la proyección existe por las formas y no se penaliza.
>
> El montaje tampoco es el mismo en lo demás: PTB tiene 929.000 tokens de entrenamiento y un vocabulario de 10.000 palabras, y el *Wikitext-2* de Inan et al. tiene 2.088.000 tokens y 33.278 palabras. Este laboratorio tiene 2.081.463 tokens y 5.003 palabras.

a) En el mínimo de cada curva, que es la comparación que el Ejercicio 3 estableció como válida, atar a igual tamaño **cuesta 5,1 puntos**. La tabla de la Nota mide lo contrario, y lo mide sin *dropout*, así que la diferencia de regularización no alcanza para explicarla. Explicá cómo pueden ser ciertas las dos afirmaciones: qué del montaje de acá —no del atado— produce la diferencia. Nombrá **dos** factores concretos y decí, para cada uno, en qué dirección esperarías que se moviera el resultado si lo cambiaras.

b) Escribí **tres** cifras que tiene que reportar una comparación entre dos arquitecturas de modelo de lenguaje para que un lector pueda decidir si le cree. Para cada una, decí qué conclusión equivocada se saca si falta, usando algo que hayas visto en este laboratorio.

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta checklist rápida:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] El `state_dict` de `atado256` no tiene ninguna clave `salida.weight`, y está impreso.
- [ ] Las cuatro llamadas a `preparar_modelo` tienen su `arq`, su `dim` y su `oculta` declarados, así que son cuatro modelos distintos y no el mismo cargado cuatro veces, y ninguna cortó por configuración.
- [ ] Respondí las preguntas de análisis de los Ejercicios 1 a 4 y el análisis final del Ejercicio 5.
- [ ] No modifiqué ninguna celda fuera de las de actividad.

---
## ¡Listo!

Comparaste cuatro modelos para medir un solo cambio de arquitectura, y el cambio resultó ser lo menos interesante del experimento. Atar las tablas ahorró el 22 % de los parámetros y costó ocho puntos de perplejidad a cinco épocas, ninguno a quince y cinco si se compara cada modelo en su mejor momento. Las tres cifras salen de las mismas cuatro corridas.

Lo que sí quedó firme es el método, y son tres reglas que se trasladan a cualquier experimento:

- **Una comparación que cambia dos cosas a la vez no mide ninguna de las dos.** El control del Ejercicio 3 no agregó una variante más: fue lo que convirtió una diferencia observada en una diferencia atribuible.
- **Una curva no es un número.** El mismo par de corridas sostiene tres conclusiones distintas según dónde se la lea, y por eso lo que se reporta es el mínimo y la época en que ocurre, no el valor al final.
- **Una sola cifra no describe un modelo.** `proyectado` alcanza la perplejidad más baja y tiene la tabla de palabras menos coherente de las cuatro; el control es el que mejor ajusta el entrenamiento y el peor en validación. Cada cifra que se omite es una conclusión equivocada que se vuelve posible.

Tres extensiones quedaron afuera, y valen la pena:

- **El atado con más vocabulario y menos corpus.** Acá hay 2.081.463 tokens de entrenamiento para 5.003 palabras, y el modelo de la clase tiene aproximadamente un parámetro por token. El montaje de *Penn Treebank* con el que se comparó en el Ejercicio 5 tiene cinco veces más parámetros por token, y es donde atar paga. Reducir el corpus o ampliar el vocabulario mueve ese cociente, y es el experimento que decidiría si los 5,1 puntos son de la técnica o de la escala.
- **El atado junto a otra regularización.** Las cuatro variantes corrieron con `DROPOUT = 0.0`. Repetir el experimento con *dropout* activo es un cambio de una línea en las celdas de preparación y cuatro entrenamientos más.
- **El corte temprano como parte del protocolo.** El Ejercicio 3 estableció el mínimo de la curva como la cifra que se reporta, pero acá se lo leyó **después** de entrenar quince épocas fijas. Cortar el entrenamiento en el momento —y decidir el corte con un tercer conjunto, no con el de validación que ya se usó para elegir— es lo que separa una comparación de laboratorio de una publicable.

Lo que sigue, en la Unidad 3, reemplaza la recurrencia por un mecanismo que calcula la representación de cada posición en función de todas las demás. La tabla de embeddings sobrevive intacta a ese cambio, y el atado de pesos también: es una de las pocas decisiones de esta unidad que los modelos de hoy siguen discutiendo en los mismos términos, y con los mismos argumentos que acabás de medir.